# 05 — Control for saccade-related overlap

This notebook estimates target-locked P3a responses using four methods and tests whether probability modulation is associated with self-functioning impairment after accounting for overlapping saccade-related activity. It produces Table S1 and the comparisons reported in the supplementary saccade-control analysis.

**Inputs:** the prepared continuous P3a FIF recordings, `eeg_features.xlsx` from notebook 02, and `erp_behavior_statistics.xlsx` from notebook 03. Finalized, manually reviewed saccade Excel files are also required. The recordings must retain `EOG1`, `EOG2`, and task annotations. The FIF files and EEG workbook are checked against the input hashes recorded in notebooks 02–03.

Study execution and export are disabled by default. Configure the paths below, enable `RUN_ANALYSES`, restart the kernel, and run all cells in order. EEG data are already preprocessed

## 1. Imports and configuration

Use the environment specified in `requirements.txt`. Regression fits operate on continuous recordings without decimation and may require substantial memory. Participants are processed one at a time.

In [ ]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import platform
import re
import warnings

import numpy as np
import pandas as pd
import mne
from scipy import stats, linalg
from mne.stats import linear_regression_raw
from IPython.display import display

SOFTWARE_VERSIONS = pd.DataFrame([
    {"package": "Python", "version": platform.python_version()},
    *[{"package": name, "version": version(name)}
      for name in ["numpy", "pandas", "scipy", "mne", "openpyxl"]],
])
display(SOFTWARE_VERSIONS)

In [ ]:
RUN_ANALYSES = False
EEG_FEATURES_FILE = Path("PATH_TO_EEG_OUTPUTS/eeg_features.xlsx").expanduser()
ERP_STATISTICS_FILE = Path("PATH_TO_STATISTICAL_OUTPUTS/erp_behavior_statistics.xlsx").expanduser()
ERP_FIF_DIR = Path("PATH_TO_PREPROCESSED_ERP_FIF").expanduser()
ERP_FILE_PATTERN = "{subject}*ica.fif"
REVIEWED_DIR = Path("PATH_TO_REVIEWED_SACCADES").expanduser()
REVIEWED_PATTERN = "{subject}_saccades.xlsx"
OUTPUT_DIR = Path("PATH_TO_OVERLAP_CONTROL_OUTPUTS").expanduser()
EXPECTED_N_PARTICIPANTS = 69

EXPORT_OUTPUTS = False
OVERWRITE_OUTPUTS = False
SAVE_EVOKEDS = False  # Optional participant-level target waveforms; keep in private storage.

EOG_ANODE = "EOG1"
EOG_CATHODE = "EOG2"
HEOG_CHANNEL = "HEOG_deconv"
SIGN_RIGHT = -1  # HEOG-step polarity corresponding to a rightward saccade.
FILTER_LOW_HZ, FILTER_HIGH_HZ = 1.0, 40.0
RESPONSE_WINDOW = (-0.4, 0.6)

DETECTOR_PARAMETERS = {
    "velocity_mad_factor": 6.0,
    "smooth_ms": 5.0,
    "refractory_ms": 80.0,
    "minimum_step_v": 30e-6,
    "step_window_ms": 50.0,
    "sign_right": SIGN_RIGHT,
}
CLEAN_LATENCY_WINDOW_MS = (90.0, 600.0)
MATCH_TOLERANCE_MS = 20.0  # Agreement diagnostic only; does not select EEG data.
REGRESSION_REJECT_TSTEP_S = 1.0
EPOCH_TMIN, EPOCH_TMAX = -0.2, 0.7
BASELINE = (-0.2, 0.0)
EEG_REJECT = {"eeg": 100e-6}  # Peak-to-peak threshold, in volts.
ERP_ROI = ["F3", "Fz", "F4", "C3", "Cz", "C4"]
ERP_WINDOW = (0.240, 0.320)

CONDITION_ORDER = ["valid_80", "invalid_80", "valid_50", "invalid_50"]
EVENT_TO_CONDITION = {
    20: "valid_80", 30: "valid_80", 21: "invalid_80", 31: "invalid_80",
    22: "valid_50", 32: "valid_50", 23: "invalid_50", 33: "invalid_50",
}
TARGET_EVENT_IDS = {condition: i + 1 for i, condition in enumerate(CONDITION_ORDER)}
SACCADE_EVENT_IDS = {"right": 99, "left": 98}
METHOD_ORDER = ["epoch_all", "epoch_detected_ge90", "regression_targets", "regression_targets_saccades"]
METHOD_LABELS = {
    "epoch_all": "Epoch averaging: all retained target trials",
    "epoch_detected_ge90": "Epoch averaging: reviewed response latency 90–600 ms",
    "regression_targets": "Continuous regression: target predictors only",
    "regression_targets_saccades": "Continuous regression: target and saccade predictors",
}
if EXPORT_OUTPUTS and not RUN_ANALYSES:
    raise ValueError("Enable RUN_ANALYSES before exporting results.")
print("Study analysis enabled:", RUN_ANALYSES)

## 2. Input validation and comparison with the primary ERP analysis

Notebook 03 supplies the self-functioning composite (`self_comp`), including missing values for participants without questionnaire scores. Scores are reused without restandardization. Each correlation uses participants with data for both variables.

The complete target-event sequence must match notebook 02. The `epoch_all` condition means, trial counts, and retained target samples are compared with notebook 02 in `Main_ERP_comparison`. Differences are reported for review; no values are substituted.

This overlap analysis preserves channel types stored in each FIF file. Notebook 02 explicitly treats `A1` and `A2` as reference channels (`misc`). If these channels remain typed as EEG in the FIF, their signals can influence rejection here. Inspect `Processing_QC` and any differences in `Main_ERP_comparison` before interpreting the control analysis. Required ROI/EOG channels cannot be marked bad, and inactive signal-space projectors must be resolved before running because epoching and regression apply them differently.

Each reviewed workbook must contain trial, event, latency_ms, and direction in its first worksheet. Latency is relative to target onset in milliseconds; direction is −1 (left), +1 (right), or 0/missing when undetermined. The complete trial and event sequence must match the EEG annotations. Missing or incomplete responses remain in the table and are excluded only from response-selected averaging. Directional correctness is not an inclusion criterion for this control.


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def normalize_subjects(frame):
    frame = frame.copy()
    if "subject" not in frame or frame["subject"].isna().any():
        raise ValueError("Nonmissing participant identifiers are required.")
    frame["subject"] = frame["subject"].astype(str).str.strip().str.upper()
    if frame["subject"].eq("").any():
        raise ValueError("Participant identifiers must not be blank.")
    return frame


def load_overlap_inputs():
    eeg_digest = sha256_file(EEG_FEATURES_FILE)
    statistics = pd.read_excel(ERP_STATISTICS_FILE,
        sheet_name=["Input_files", "Participant_ERP"], engine="openpyxl")
    if eeg_digest not in set(statistics["Input_files"]["sha256"].astype(str)):
        raise ValueError("The EEG workbook differs from the input used by notebook 03. Run 03 with this file first.")
    participant = normalize_subjects(statistics["Participant_ERP"])
    required = {"self_comp", "probability_modulation_uv", *[f"erp_{c}_uv" for c in CONDITION_ORDER]}
    if missing := required - set(participant):
        raise ValueError(f"Missing Participant_ERP columns: {sorted(missing)}.")
    if participant["subject"].duplicated().any() or len(participant) != EXPECTED_N_PARTICIPANTS:
        raise ValueError("Participant_ERP must contain the complete configured cohort, once per participant.")
    participant["self_comp"] = pd.to_numeric(participant["self_comp"], errors="raise")
    if np.isinf(participant["self_comp"]).any():
        raise ValueError("Self-functioning scores must be finite or missing.")
    tables = pd.read_excel(EEG_FEATURES_FILE,
        sheet_name=["ERP_trials", "Event_alignment", "EEG_input_QC"], engine="openpyxl")
    for name in tables:
        tables[name] = normalize_subjects(tables[name])
    trials, alignment = tables["ERP_trials"], tables["Event_alignment"]
    if {"target_sample", "condition", "erp_amplitude_uv"} - set(trials):
        raise ValueError("The ERP trial table is incomplete.")
    if {"target_sample", "event", "trial"} - set(alignment):
        raise ValueError("The full target-event alignment table is incomplete.")
    for name, table in [("ERP_trials", trials), ("Event_alignment", alignment)]:
        if set(table["subject"]) != set(participant["subject"]):
            raise ValueError(f"Cohort mismatch in {name}.")
        if table.duplicated(["subject", "target_sample"]).any():
            raise ValueError(f"Duplicate participant/target samples in {name}.")
    if not trials["condition"].isin(CONDITION_ORDER).all():
        raise ValueError("Unknown ERP condition labels.")
    for table, columns in [(trials,["erp_amplitude_uv","target_sample"]), (alignment,["target_sample","event","trial"])]:
        for column in columns:
            table[column] = pd.to_numeric(table[column], errors="raise")
            if not np.isfinite(table[column]).all():
                raise ValueError(f"Nonfinite input values in {column}.")
    reference = trials.groupby(["subject", "condition"])["erp_amplitude_uv"].agg(
        reference_mean_uv="mean", reference_n_trials="size").reset_index()
    reference_wide = reference.pivot(index="subject", columns="condition", values="reference_mean_uv").reindex(
        index=participant["subject"], columns=CONDITION_ORDER)
    if reference_wide.isna().any(axis=None):
        raise ValueError("Every participant needs ERP data for all four conditions.")
    for condition in CONDITION_ORDER:
        if not np.allclose(reference_wide[condition], participant[f"erp_{condition}_uv"], rtol=1e-10, atol=1e-10):
            raise ValueError("ERP means in notebooks 02 and 03 disagree.")
    reference_modulation = ((reference_wide["invalid_80"] - reference_wide["valid_80"])
                            - (reference_wide["invalid_50"] - reference_wide["valid_50"]))
    if not np.allclose(reference_modulation, participant["probability_modulation_uv"], rtol=1e-10, atol=1e-10):
        raise ValueError("The probability contrast in notebook 03 is inconsistent with its condition means.")
    metadata = tables["EEG_input_QC"].loc[tables["EEG_input_QC"]["analysis"].eq("P3a")].copy()
    if metadata["subject"].duplicated().any() or set(metadata["subject"]) != set(participant["subject"]):
        raise ValueError("EEG_input_QC must identify one P3a recording per participant.")
    if {"sha256", "sfreq_hz", "first_samp", "n_samples"} - set(metadata):
        raise ValueError("Recording identity and sampling metadata are required from notebook 02.")
    input_files = pd.DataFrame([
        {"role": "EEG features", "file": EEG_FEATURES_FILE.name, "sha256": eeg_digest},
        {"role": "ERP statistics", "file": ERP_STATISTICS_FILE.name, "sha256": sha256_file(ERP_STATISTICS_FILE)},
    ])
    return participant.sort_values("subject"), trials, alignment, metadata, reference, input_files


def resolve_erp_recording(subject):
    if "{subject}" not in ERP_FILE_PATTERN:
        raise ValueError("ERP_FILE_PATTERN must include {subject}.")
    paths = sorted(path for path in ERP_FIF_DIR.glob(ERP_FILE_PATTERN.format(subject=subject)) if path.is_file())
    if len(paths) != 1:
        raise ValueError(f"{subject}: expected exactly one matching P3a FIF recording; found {len(paths)}.")
    return paths[0]


def compare_main_erp(result, subject, reference_trials, reference_summary):
    main = result["condition_amplitudes"].loc[
        result["condition_amplitudes"]["method"].eq("epoch_all")].copy()
    main = main.merge(reference_summary, on=["subject", "condition"], validate="one_to_one")
    main["mean_difference_uv"] = main["erp_amplitude_uv"] - main["reference_mean_uv"]
    main["count_difference"] = main["n_events"] - main["reference_n_trials"]
    main["amplitude_agrees"] = np.isclose(main["erp_amplitude_uv"], main["reference_mean_uv"], rtol=1e-6, atol=1e-6)
    for index, row in main.iterrows():
        source_samples = set(reference_trials.loc[reference_trials["condition"].eq(row["condition"]), "target_sample"])
        qc = result["trial_qc"]
        current_samples = set(qc.loc[qc["condition"].eq(row["condition"]) & qc["epoch_all_retained"], "target_sample"])
        main.loc[index, "retained_samples_agree"] = source_samples == current_samples
        main.loc[index, "n_only_in_main_erp"] = len(source_samples - current_samples)
        main.loc[index, "n_only_in_overlap_control"] = len(current_samples - source_samples)
    return main


def read_reviewed(subject, expected_codes):
    """Read final manual annotations without changing trial identities or values."""
    if "{subject}" not in REVIEWED_PATTERN:
        raise ValueError("REVIEWED_PATTERN must include {subject}.")
    path = REVIEWED_DIR / REVIEWED_PATTERN.format(subject=subject)
    table = pd.read_excel(path, sheet_name=0, engine="openpyxl").dropna(how="all")
    required = ["trial", "event", "latency_ms", "direction"]
    if missing := set(required) - set(table):
        raise ValueError(f"{subject}: missing reviewed-annotation columns: {sorted(missing)}.")
    table = table[required].copy()
    for column in required:
        table[column] = pd.to_numeric(table[column], errors="raise")
    for column in ["trial", "event"]:
        values = table[column].to_numpy(dtype=float)
        if not np.isfinite(values).all() or not np.equal(values, np.floor(values)).all():
            raise ValueError(f"{subject}: {column} must contain finite integer values.")
    table = table.sort_values("trial").reset_index(drop=True)
    if not np.array_equal(table["trial"].to_numpy(), np.arange(len(expected_codes))):
        raise ValueError(f"{subject}: reviewed trial IDs must match the complete EEG target sequence.")
    if not np.array_equal(table["event"].to_numpy(), expected_codes):
        raise ValueError(f"{subject}: reviewed event codes differ from the EEG event order.")
    if np.isinf(table[["latency_ms", "direction"]].to_numpy(dtype=float)).any():
        raise ValueError(f"{subject}: reviewed latency and direction cannot be infinite.")
    if not table["direction"].dropna().isin([-1, 0, 1]).all():
        raise ValueError(f"{subject}: reviewed direction must be -1, 0, +1, or missing.")
    return table, {"role": "Final reviewed saccades", "subject": subject,
                   "file": path.name, "sha256": sha256_file(path)}

## 3. Continuous saccade detection and reviewed responses

The continuous detector and its parameters are identical to notebook 00. Construct HEOG as EOG1 − EOG2, filter it at 1–40 Hz, smooth it with a 5-ms moving average, and detect upward crossings of the recording-wide absolute-velocity threshold. The threshold is median absolute velocity + 6 × 1.4826 × (MAD + 10⁻¹² V/s). Accepted onsets are at least 80 ms apart and have an absolute HEOG step of at least 30 µV, measured using 50-ms pre- and post-onset windows. Directions returned by the detector are physical left (−1) or right (+1).

All continuous detections enter the deconvolution, including movements outside the target response window. The first automatic event from −400 through +600 ms is reported separately. Agreement within 20 ms is assessed using the nearest continuous detection to each reviewed onset. Automatic response selection and the 20-ms matching tolerance do not determine which epochs enter response-selected averaging.

Response-selected averaging uses the final reviewed latency and direction: both must be defined, direction must be −1 or +1, and latency must be 90–600 ms, including both endpoints. Correctly and incorrectly directed responses are eligible. This differs from CNV exclusion, which concerns pretarget responses.

Target events stay fixed. If a saccade predictor coincides with another event, it is moved to the next free sample and the shift is recorded. Review comparisons use the unshifted onset.

In [ ]:
def detect_saccades_continuous(
    signal, sfreq, *, velocity_mad_factor=6.0, smooth_ms=5.0,
    refractory_ms=80.0, minimum_step_v=30e-6, step_window_ms=50.0,
    sign_right=-1,
):
    """Detect all qualifying onsets in continuous, filtered HEOG.

    Returns a dictionary with onset_indices, directions, velocity_v_s,
    and velocity_threshold_v_s. Onset indices are zero-based within
    signal; directions are physical left (-1) or right (+1).
    """
    signal = np.asarray(signal, dtype=float)
    if signal.ndim != 1 or signal.size < 2 or not np.isfinite(signal).all():
        raise ValueError("HEOG must be a finite one-dimensional signal with at least two samples.")
    parameters = [sfreq, velocity_mad_factor, smooth_ms, refractory_ms,
                  minimum_step_v, step_window_ms]
    if not np.isfinite(parameters).all() or any(value <= 0 for value in parameters):
        raise ValueError("Sampling rate and detector parameters must be positive and finite.")
    if sign_right not in (-1, 1):
        raise ValueError("sign_right must be -1 or +1.")

    smoothing_samples = max(1, int(round(smooth_ms / 1000 * sfreq)))
    step_samples = int(step_window_ms / 1000 * sfreq)
    refractory_samples = int(refractory_ms / 1000 * sfreq)
    if smoothing_samples > signal.size or step_samples < 1:
        raise ValueError("The signal length or sampling rate is incompatible with the detector windows.")
    smoothed = np.convolve(
        signal, np.ones(smoothing_samples) / smoothing_samples, mode="same",
    )
    velocity = np.gradient(smoothed, 1.0 / sfreq)
    absolute_velocity = np.abs(velocity)
    median_velocity = np.median(absolute_velocity)
    velocity_mad = np.median(np.abs(absolute_velocity - median_velocity)) + 1e-12
    threshold = median_velocity + velocity_mad_factor * 1.4826 * velocity_mad
    above_threshold = absolute_velocity > threshold
    candidates = np.flatnonzero(~above_threshold[:-1] & above_threshold[1:]) + 1

    onsets, directions = [], []
    last_onset = -10**9
    for onset in candidates:
        if onset - last_onset < refractory_samples:
            continue
        pre_level = np.median(smoothed[max(0, onset - step_samples):onset])
        post_level = (
            np.median(smoothed[onset:onset + step_samples])
            if onset + step_samples < len(smoothed) else smoothed[-1]
        )
        step = post_level - pre_level
        if abs(step) >= minimum_step_v:
            onsets.append(onset)
            directions.append(sign_right if step > 0 else -sign_right)
            last_onset = onset

    return {
        "onset_indices": np.asarray(onsets, dtype=int),
        "directions": np.asarray(directions, dtype=int),
        "velocity_v_s": velocity,
        "velocity_threshold_v_s": float(threshold),
    }


def combine_target_saccade_events(target_events, saccade_samples, directions, last_sample):
    """Keep targets fixed and move a coincident saccade to the next free sample."""
    targets = np.asarray(target_events, dtype=int)
    samples = np.asarray(saccade_samples, dtype=int)
    directions = np.asarray(directions, dtype=int)
    if len(np.unique(targets[:, 0])) != len(targets):
        raise ValueError("Duplicate target samples: resolve the annotation ambiguity before regression.")
    if len(samples) != len(directions) or not np.isin(directions, [-1, 1]).all():
        raise ValueError("Each saccade onset requires a physical direction of -1 or +1.")
    if samples.size and (np.diff(samples) <= 0).any():
        raise ValueError("Saccade samples must be unique and in increasing order.")
    occupied = set(targets[:, 0].tolist())
    used_samples = samples.copy()
    for i, onset in enumerate(samples):
        used = int(onset)
        while used in occupied:
            used += 1
        if used > last_sample:
            raise ValueError("Resolving an event collision would place a saccade beyond the recording.")
        used_samples[i] = used
        occupied.add(used)
    codes = np.where(directions > 0, SACCADE_EVENT_IDS["right"],
                     SACCADE_EVENT_IDS["left"])
    saccades = np.column_stack([used_samples, np.zeros(len(samples), dtype=int), codes])
    combined = np.vstack([targets, saccades])
    combined = combined[np.argsort(combined[:, 0], kind="stable")]
    return combined, used_samples

## 4. Four target-response estimates and artifact exclusion

All estimates use −200 to +700 ms relative to the target and baseline correction from −200 to 0 ms. The outcome is mean amplitude over the six ROI electrodes and 240–320 ms, including both endpoints.

| Method | Target-response estimate |
|---|---|
| epoch_all | Average all artifact-retained target epochs. |
| epoch_detected_ge90 | Average artifact-retained epochs with a reviewed response at 90–600 ms and a defined direction. |
| regression_targets | Fit four target event types to the continuous recording. |
| regression_targets_saccades | Add separate leftward and rightward saccade event types. |

Epoch averaging rejects epochs overlapping BAD annotations or exceeding 100 µV peak-to-peak on a usable EEG channel. Continuous regression excludes BAD samples and full, nonoverlapping 1-s segments exceeding the same peak-to-peak threshold. An incomplete final segment is not screened by the amplitude criterion; any BAD samples in it are still excluded. Zero-duration BAD annotations define no continuous interval to mask; their count is reported.

Both regression models use one shared sample mask, computed before fitting. Time stamps and all event predictors remain intact, including events whose onsets fall in excluded intervals: their contributions outside those intervals can still be estimated. Rejected observations are marked as missing in a private EEG copy and removed by the solver before matrix multiplication. The recording is never shortened or concatenated. Samples with no active predictor are omitted internally by MNE; they cannot affect the fitted coefficients.

The regressions use MNE time-expanded predictors over −200 to +700 ms for every event type, no decimation, and a masked Cholesky solver. Estimated target waveforms are baseline-corrected after fitting. Both directions and all four conditions are required. Fitting errors stop the analysis.

In Condition_amplitudes, n_events is the retained epoch count for averages and the number of supplied events before sample exclusion for regression. For regression it is not an effective sample size or a count of accepted epochs.

In [ ]:
def prepare_regression_data(raw):
    """Create one EEG copy and artifact mask shared by both regression models."""
    eeg_picks = mne.pick_types(raw.info, meg=False, eeg=True, exclude="bads")
    if not len(eeg_picks):
        raise ValueError("No usable EEG channels for regression.")
    regression_raw = raw.copy().pick(eeg_picks).load_data()
    signal = regression_raw.get_data()
    if not np.isfinite(signal).all():
        raise ValueError("The input EEG contains nonfinite samples before artifact masking.")

    # MNE maps annotation times to the sampling grid, including first_samp/orig_time.
    annotation_signal = regression_raw.get_data(
        picks=[0], reject_by_annotation="NaN", verbose="error",
    )[0]
    bad_annotation = np.isnan(annotation_signal)
    amplitude_rejected = np.zeros(regression_raw.n_times, dtype=bool)
    step = int(np.ceil(REGRESSION_REJECT_TSTEP_S * regression_raw.info["sfreq"]))
    if step < 1:
        raise ValueError("The amplitude-screening window must contain at least one sample.")
    n_rejected_segments = 0
    for start in range(0, regression_raw.n_times - step + 1, step):
        stop = start + step
        if np.any(np.ptp(signal[:, start:stop], axis=1) > EEG_REJECT["eeg"]):
            amplitude_rejected[start:stop] = True
            n_rejected_segments += 1
    excluded = bad_annotation | amplitude_rejected
    if excluded.all():
        raise ValueError("Artifact rejection removed every EEG sample.")

    def mark_excluded(values):
        values[:, excluded] = np.nan
        return values

    regression_raw.apply_function(mark_excluded, picks="all", channel_wise=False, verbose="error")
    qc = {
        "n_regression_bad_samples": int(bad_annotation.sum()),
        "n_regression_amplitude_samples": int(amplitude_rejected.sum()),
        "n_regression_excluded_samples": int(excluded.sum()),
        "n_regression_eligible_samples": int((~excluded).sum()),
        "n_regression_rejected_segments": n_rejected_segments,
        "n_regression_unchecked_tail_samples": int(regression_raw.n_times % step),
    }
    return regression_raw, qc


def masked_cholesky(design, observations):
    """Fit ordinary least squares after excluding marked observation rows."""
    keep = np.isfinite(observations).all(axis=1)
    if not keep.any():
        raise ValueError("No usable observations remain within the regression predictor windows.")
    design = design[keep]
    observations = observations[keep]
    gram = (design.T @ design).toarray()
    return linalg.solve(
        gram, design.T @ observations, assume_a="pos",
        overwrite_a=True, overwrite_b=True,
    ).T


def reviewed_response_qc(reviewed, targets, absolute_onsets, directions, sfreq, subject):
    """Compare final manual responses with automatic detections for reporting."""
    latency = reviewed["latency_ms"].to_numpy(dtype=float)
    direction = reviewed["direction"].to_numpy(dtype=float)
    detected = np.isfinite(latency) & np.isin(direction, [-1, 1])
    incomplete = np.isfinite(latency) != np.isin(direction, [-1, 1])
    lower, upper = CLEAN_LATENCY_WINDOW_MS
    selected = detected & (latency >= lower) & (latency <= upper)
    reason = np.select(
        [incomplete, ~detected, latency < lower, latency > upper],
        ["Incomplete annotation", "No reviewed response", "Latency below lower limit", "Latency above upper limit"],
        default="Included",
    )
    rows = []
    for i, target in enumerate(targets[:, 0]):
        relative_ms = (absolute_onsets - target) * 1000.0 / sfreq
        candidates = np.flatnonzero(
            (relative_ms >= RESPONSE_WINDOW[0] * 1000 - 1e-9)
            & (relative_ms <= RESPONSE_WINDOW[1] * 1000 + 1e-9)
        )
        automatic_latency = float(relative_ms[candidates[0]]) if candidates.size else np.nan
        automatic_direction = int(directions[candidates[0]]) if candidates.size else 0
        nearest_sample, nearest_direction, onset_error = np.nan, np.nan, np.nan
        if detected[i] and absolute_onsets.size:
            expected_sample = target + latency[i] * sfreq / 1000.0
            nearest = int(np.argmin(np.abs(absolute_onsets - expected_sample)))
            nearest_sample = int(absolute_onsets[nearest])
            nearest_direction = int(directions[nearest])
            onset_error = abs(nearest_sample - expected_sample) * 1000.0 / sfreq
        rows.append({
            "subject": subject, "trial": i,
            "reviewed_latency_ms": latency[i], "reviewed_direction": direction[i],
            "reviewed_detected": bool(detected[i]), "incomplete_annotation": bool(incomplete[i]),
            "response_selected": bool(selected[i]), "response_exclusion_reason": reason[i],
            "automatic_latency_ms": automatic_latency, "automatic_direction": automatic_direction,
            "automatic_response_selected": bool(np.isfinite(automatic_latency) and lower <= automatic_latency <= upper),
            "nearest_auto_sample": nearest_sample, "nearest_auto_direction": nearest_direction,
            "onset_error_ms": onset_error, "matched_within_tolerance": bool(onset_error <= MATCH_TOLERANCE_MS),
            "matched_direction_disagrees": bool(onset_error <= MATCH_TOLERANCE_MS and nearest_direction != direction[i]),
        })
    table = pd.DataFrame(rows)
    matched = table["matched_within_tolerance"]
    table["shared_matched_auto_event"] = False
    table.loc[matched, "shared_matched_auto_event"] = table.loc[matched, "nearest_auto_sample"].duplicated(keep=False)
    return table, selected


def _target_roi_amplitude(evoked, config):
    missing = set(config["roi"]) - set(evoked.ch_names)
    if missing:
        raise ValueError(f"Missing fixed-ROI channels in estimated response: {sorted(missing)}")
    values = evoked.copy().pick(list(config["roi"]))
    lower, upper = config["roi_window_s"]
    mask = (values.times >= lower - 1e-9) & (values.times <= upper + 1e-9)
    if not mask.any():
        raise ValueError("The ROI time window contains no samples.")
    return float(values.data[:, mask].mean() * 1e6)


def _epoch_target_qc(epochs, target_trial_indices, n_targets):
    retained = np.zeros(n_targets, dtype=bool)
    reasons = [""] * n_targets
    for local_index, source_trial in enumerate(target_trial_indices):
        reasons[int(source_trial)] = "; ".join(epochs.drop_log[local_index])
    retained[np.asarray(target_trial_indices, dtype=int)[epochs.selection]] = True
    return retained, reasons


def _extract_four_methods(raw, subject):
    """Estimate four target ERP variants for one participant.

    Returns DataFrames for amplitudes, contrasts, target and saccade QC,
    processing settings, and a dictionary of target-locked Evoked objects.
    The continuous input must already have the preprocessing used for the ERP
    analysis. No additional EEG filtering, resampling, or ICA is performed.
    """
    cfg = {
        "eog_anode": EOG_ANODE, "eog_cathode": EOG_CATHODE, "heog_name": HEOG_CHANNEL,
        "heog_l_freq": FILTER_LOW_HZ, "heog_h_freq": FILTER_HIGH_HZ,
        "epoch_tmin": EPOCH_TMIN, "epoch_tmax": EPOCH_TMAX,
        "baseline": BASELINE, "reject_eeg_v": EEG_REJECT["eeg"],
        "regression_tstep_s": REGRESSION_REJECT_TSTEP_S, "roi": ERP_ROI, "roi_window_s": ERP_WINDOW,
    }
    if SIGN_RIGHT not in (-1, 1):
        raise ValueError("SIGN_RIGHT must be -1 or +1.")
    _description_codes = {str(event): TARGET_EVENT_IDS[condition]
                          for event, condition in EVENT_TO_CONDITION.items()}
    missing = set(cfg["roi"]) | {cfg["eog_anode"], cfg["eog_cathode"]}
    missing -= set(raw.ch_names)
    if missing:
        raise ValueError(f"{subject}: required channels are missing: {sorted(missing)}")
    bad_required = (set(cfg["roi"]) | {cfg["eog_anode"], cfg["eog_cathode"]}) & set(raw.info["bads"])
    if bad_required:
        raise ValueError(f"{subject}: required channels are marked bad: {sorted(bad_required)}")
    if any(not projection["active"] for projection in raw.info["projs"]):
        raise ValueError(f"{subject}: inactive signal-space projectors would be applied differently by epoching and regression.")
    if cfg["heog_name"] in raw.ch_names:
        raise ValueError(f"{subject}: choose an unused bipolar HEOG channel name.")
    if cfg["heog_h_freq"] >= raw.info["sfreq"] / 2:
        raise ValueError(f"{subject}: the HEOG upper cutoff must be below the Nyquist frequency.")
    data = raw.copy().load_data()
    data = mne.set_bipolar_reference(data, cfg["eog_anode"], cfg["eog_cathode"],
                                    ch_name=cfg["heog_name"], drop_refs=False, verbose="error")
    data.set_channel_types({cfg["heog_name"]: "eog"}, verbose="error")
    sfreq = float(data.info["sfreq"])
    annotation_events, annotation_ids = mne.events_from_annotations(data, verbose="error")
    code_to_description = {value: key for key, value in annotation_ids.items()}
    selected = [(int(sample), code_to_description[code]) for sample, _, code in annotation_events
                if code_to_description.get(code) in _description_codes]
    if not selected:
        raise ValueError(f"{subject}: no recognized target annotations.")
    targets = np.asarray([[sample, 0, _description_codes[description]]
                          for sample, description in selected], dtype=int)
    if len(np.unique(targets[:, 0])) != len(targets):
        raise ValueError(f"{subject}: target annotations contain duplicate samples.")
    if set(targets[:, 2]) != set(TARGET_EVENT_IDS.values()):
        raise ValueError(f"{subject}: all four target conditions are required.")
    if (targets[:, 0] < data.first_samp).any() or (targets[:, 0] > data.last_samp).any():
        raise ValueError(f"{subject}: target event samples fall outside the recording.")

    filtered_heog = data.copy().pick([cfg["heog_name"]]).filter(
        cfg["heog_l_freq"], cfg["heog_h_freq"], picks="all", verbose="error")
    detections = detect_saccades_continuous(
        filtered_heog.get_data()[0], sfreq, **DETECTOR_PARAMETERS,
    )
    onsets = detections["onset_indices"]
    directions = detections["directions"]
    absolute_onsets = data.first_samp + onsets
    if set(directions) != {-1, 1}:
        raise ValueError(f"{subject}: both saccade directions are required for regression.")
    reviewed, reviewed_file = read_reviewed(subject, [int(description) for _, description in selected])
    review_qc, response_selected = reviewed_response_qc(
        reviewed, targets, absolute_onsets, directions, sfreq, subject,
    )
    if set(targets[response_selected, 2]) != set(TARGET_EVENT_IDS.values()):
        raise ValueError(f"{subject}: the selected-response subset lacks a target condition.")
    epoch_arguments = dict(event_id=TARGET_EVENT_IDS, tmin=cfg["epoch_tmin"],
                           tmax=cfg["epoch_tmax"], baseline=cfg["baseline"],
                           reject={"eeg": cfg["reject_eeg_v"]}, event_repeated="drop",
                           reject_by_annotation=True, preload=True, verbose="error")
    epochs_all = mne.Epochs(data, targets, **epoch_arguments)
    epochs_selected = mne.Epochs(data, targets[response_selected], **epoch_arguments)
    for label, epochs in [("epoch_all", epochs_all), ("epoch_detected_ge90", epochs_selected)]:
        if any(len(epochs[condition]) == 0 for condition in TARGET_EVENT_IDS):
            raise ValueError(f"{subject}: {label} has a target condition with no retained epochs.")
    combined_events, regression_onsets = combine_target_saccade_events(
        targets, absolute_onsets, directions, data.last_samp)
    event_ids = dict(TARGET_EVENT_IDS, **SACCADE_EVENT_IDS)
    regression_data, artifact_qc = prepare_regression_data(data)
    regression_arguments = dict(tmin=cfg["epoch_tmin"], tmax=cfg["epoch_tmax"],
                                reject=None,
                                tstep=cfg["regression_tstep_s"], decim=1,
                                solver=masked_cholesky, picks=None)
    regression_saccades = linear_regression_raw(
        regression_data, combined_events, event_id=event_ids, **regression_arguments)
    regression_targets = linear_regression_raw(
        regression_data, targets, event_id=TARGET_EVENT_IDS, **regression_arguments)
    evokeds = {
        "epoch_all": {condition: epochs_all[condition].average() for condition in TARGET_EVENT_IDS},
        "epoch_detected_ge90": {condition: epochs_selected[condition].average() for condition in TARGET_EVENT_IDS},
        "regression_targets": {condition: regression_targets[condition].apply_baseline(cfg["baseline"], verbose="error")
                               for condition in TARGET_EVENT_IDS},
        "regression_targets_saccades": {condition: regression_saccades[condition].apply_baseline(cfg["baseline"], verbose="error")
                                        for condition in TARGET_EVENT_IDS},
    }
    amplitude_rows, contrast_rows = [], []
    for method, condition_evokeds in evokeds.items():
        amplitudes = {condition: _target_roi_amplitude(evoked, cfg)
                      for condition, evoked in condition_evokeds.items()}
        for condition, amplitude in amplitudes.items():
            amplitude_rows.append({"subject": subject, "method": method, "condition": condition,
                                   "erp_amplitude_uv": amplitude,
                                   "n_events": int(condition_evokeds[condition].nave)})
        validity_80 = amplitudes["invalid_80"] - amplitudes["valid_80"]
        validity_50 = amplitudes["invalid_50"] - amplitudes["valid_50"]
        contrast_rows.append({"subject": subject, "method": method,
                              "validity_effect_80_uv": validity_80,
                              "validity_effect_50_uv": validity_50,
                              "probability_modulation_uv": validity_80 - validity_50})
    kept_all, reasons_all = _epoch_target_qc(epochs_all, np.arange(len(targets)), len(targets))
    kept_selected, reasons_selected = _epoch_target_qc(epochs_selected, np.flatnonzero(response_selected), len(targets))
    trial_qc = pd.DataFrame({
        "subject": subject, "trial": np.arange(len(targets)), "target_sample": targets[:, 0],
        "event": [int(description) for _, description in selected],
        "condition": [next(name for name, value in TARGET_EVENT_IDS.items() if value == code) for code in targets[:, 2]],
        "epoch_all_retained": kept_all, "epoch_all_drop_reason": reasons_all,
        "epoch_detected_ge90_retained": kept_selected,
        "epoch_detected_ge90_drop_reason": [reason if selected else "Response criterion not met"
                                             for reason, selected in zip(reasons_selected, response_selected)],
    })
    trial_qc = trial_qc.merge(review_qc, on=["subject", "trial"], validate="one_to_one")
    saccade_qc = pd.DataFrame({
        "subject": subject, "onset_sample_relative": onsets,
        "onset_sample_absolute": absolute_onsets,
        "regression_sample": regression_onsets,
        "regression_shift_samples": regression_onsets - absolute_onsets,
        "direction": np.where(directions > 0, "right", "left"),
        "direction_code": directions,
    })
    processing_qc = pd.DataFrame([{
        "subject": subject, "sfreq_hz": sfreq, "first_samp": data.first_samp,
        "n_recording_samples": data.n_times, "n_targets": len(targets),
        "n_continuous_saccades": len(onsets), "n_rightward_saccades": int((directions > 0).sum()),
        "n_leftward_saccades": int((directions < 0).sum()),
        "n_incomplete_reviewed_annotations": int(review_qc["incomplete_annotation"].sum()),
        "n_selected_responses_before_epoch_rejection": int(response_selected.sum()),
        "selected_response_fraction": float(response_selected.mean()),
        "n_epochs_all": int(len(epochs_all)), "n_epochs_selected": int(len(epochs_selected)),
        "n_shifted_saccade_predictors": int((regression_onsets != absolute_onsets).sum()),
        "maximum_predictor_shift_samples": int(np.max(regression_onsets - absolute_onsets)),
        "n_bad_annotations": sum(str(description).lower().startswith("bad") for description in data.annotations.description),
        "regression_reject_tstep_s": cfg["regression_tstep_s"],
        "regression_reject_by_annotation": True,
        "epoch_reject_by_annotation": True,
        "roi_channels": ", ".join(cfg["roi"]),
        "eeg_channels": ", ".join(data.copy().pick("eeg", exclude=[]).ch_names),
        "channel_types": "; ".join(f"{name}: {kind}" for name, kind in zip(data.ch_names, data.get_channel_types())),
        "bad_channels": ", ".join(data.info["bads"]),
        "heog_anode": EOG_ANODE, "heog_cathode": EOG_CATHODE,
        "heog_l_freq_hz": FILTER_LOW_HZ, "heog_h_freq_hz": FILTER_HIGH_HZ,
        "heog_sign_right": SIGN_RIGHT,
        "minimum_response_latency_ms": CLEAN_LATENCY_WINDOW_MS[0],
        "maximum_response_latency_ms": CLEAN_LATENCY_WINDOW_MS[1],
        "minimum_heog_step_uv": DETECTOR_PARAMETERS["minimum_step_v"] * 1e6,
        "velocity_mad_factor": DETECTOR_PARAMETERS["velocity_mad_factor"],
        "eeg_reject_peak_to_peak_uv": EEG_REJECT["eeg"] * 1e6,
        "regression_decimation": 1, "regression_solver": "masked_cholesky",
        "input_highpass_hz": raw.info["highpass"], "input_lowpass_hz": raw.info["lowpass"],
        "n_zero_duration_bad_annotations": sum(
            str(description).lower().startswith("bad") and duration == 0
            for description, duration in zip(data.annotations.description, data.annotations.duration)
        ),
        "velocity_threshold_v_s": detections["velocity_threshold_v_s"],
        **artifact_qc,
    }])
    return {
        "condition_amplitudes": pd.DataFrame(amplitude_rows), "participant_contrasts": pd.DataFrame(contrast_rows),
        "trial_qc": trial_qc, "saccade_events": saccade_qc,
        "processing_qc": processing_qc, "reviewed_file": reviewed_file, "evokeds": {method: [condition_map[condition] for condition in CONDITION_ORDER]
                                                    for method, condition_map in evokeds.items()},
    }


def analyze_overlap_participant(raw, subject):
    """Run the four estimators and retain all emitted warning messages."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = _extract_four_methods(raw, subject)
    result["warnings"] = pd.DataFrame(
        [{"subject": subject, "category": item.category.__name__, "message": str(item.message)}
         for item in caught], columns=["subject", "category", "message"])
    return result

## 5. Participant contrasts and statistical comparisons

For each method, calculate the invalid-minus-valid effect separately in the 80% and 50% contexts. Probability modulation is the 80% effect minus the 50% effect.

Table S1 reports two-sided Spearman correlations between probability modulation and self-functioning impairment for all four methods. Method agreement compares `epoch_all` with the response-selected and deconvolved estimates. Group comparisons use two-sided Wilcoxon signed-rank tests for the 80% versus 50% validity effects, separately for epoch averaging, target-only regression, and deconvolution (`zero_method="wilcox"`, no continuity correction, `method="auto"`). These supplementary checks use unadjusted p values; they do not fit additional mixed models.

In [ ]:
def spearman_summary(x, y):
    frame = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    enough = len(frame) >= 3 and frame["x"].nunique() > 1 and frame["y"].nunique() > 1
    if not enough:
        return {"n": len(frame), "rho": np.nan, "p": np.nan, "status": "insufficient_or_constant"}
    rho, p = stats.spearmanr(frame["x"], frame["y"])
    return {"n": len(frame), "rho": rho, "p": p, "status": "ok"}


def overlap_statistics(contrasts, participant):
    joined = contrasts.merge(participant[["subject", "self_comp"]], on="subject", validate="many_to_one")
    correlations = []
    for number, method in enumerate(METHOD_ORDER, 1):
        group = joined.loc[joined["method"].eq(method)]
        correlations.append({"method_number": number, "method": method, "description": METHOD_LABELS[method],
            **spearman_summary(group["probability_modulation_uv"], group["self_comp"])})
    wide = contrasts.pivot(index="subject", columns="method", values="probability_modulation_uv")
    agreement = pd.DataFrame([
        {"method_a": "epoch_all", "method_b": method,
         **spearman_summary(wide["epoch_all"], wide[method])}
        for method in ["epoch_detected_ge90", "regression_targets_saccades"]
    ])
    group_rows = []
    for method in ["epoch_all", "regression_targets", "regression_targets_saccades"]:
        group = contrasts.loc[contrasts["method"].eq(method), ["validity_effect_80_uv", "validity_effect_50_uv"]].dropna()
        differences = group["validity_effect_80_uv"] - group["validity_effect_50_uv"]
        n_nonzero = int(differences.ne(0).sum())
        if len(group) and n_nonzero == 0:
            w, p, status = 0., 1., "all_differences_zero"
        elif n_nonzero >= 3:
            test = stats.wilcoxon(group["validity_effect_80_uv"], group["validity_effect_50_uv"],
                                 alternative="two-sided", zero_method="wilcox", correction=False, method="auto")
            w, p, status = test.statistic, test.pvalue, "ok"
        else:
            w, p, status = np.nan, np.nan, "insufficient_nonzero_pairs"
        group_rows.append({"method": method, "n": len(group), "n_nonzero": n_nonzero,
            "median_modulation_uv": differences.median(), "W": w, "p": p, "status": status})
    return pd.DataFrame(correlations), agreement, pd.DataFrame(group_rows), joined

## 6. Run the cohort analysis

No participant is silently excluded after a processing error. Verify file patterns, event alignment, channel information, and model warnings before using the results. Optional participant waveform exports require `SAVE_EVOKEDS=True` and are retained in memory until export.

In [ ]:
def run_overlap_analysis():
    participant, reference_trials, alignment, metadata, reference_summary, input_files = load_overlap_inputs()
    collected = {key: [] for key in ["condition_amplitudes", "participant_contrasts", "trial_qc",
                                    "saccade_events", "processing_qc", "warnings"]}
    comparisons, recording_files, evoked_outputs = [], [], {}
    meta_index = metadata.set_index("subject")
    for position, subject in enumerate(participant["subject"], 1):
        if not re.fullmatch(r"[A-Z0-9][A-Z0-9_-]*", subject):
            raise ValueError(f"Participant identifier is not filename-safe: {subject!r}.")
        path = resolve_erp_recording(subject)
        digest = sha256_file(path)
        expected = meta_index.loc[subject]
        if digest != str(expected["sha256"]):
            raise ValueError(f"{subject}: FIF content differs from the recording used by notebook 02.")
        raw = mne.io.read_raw_fif(path, preload=True, verbose="error")
        try:
            if (not np.isclose(raw.info["sfreq"], expected["sfreq_hz"], rtol=0, atol=1e-9)
                    or raw.first_samp != expected["first_samp"] or raw.n_times != expected["n_samples"]):
                raise ValueError(f"{subject}: recording sampling metadata disagree with notebook 02.")
            with mne.use_log_level("ERROR"):
                result = analyze_overlap_participant(raw, subject)
        finally:
            raw.close()
        expected_events = alignment.loc[alignment["subject"].eq(subject)].sort_values("trial")
        current_events = result["trial_qc"].sort_values("trial")
        for column in ["trial", "event", "target_sample"]:
            if not np.array_equal(expected_events[column].to_numpy(), current_events[column].to_numpy()):
                raise ValueError(f"{subject}: complete target sequence differs from notebook 02 ({column}).")
        comparisons.append(compare_main_erp(
            result, subject,
            reference_trials.loc[reference_trials["subject"].eq(subject)],
            reference_summary.loc[reference_summary["subject"].eq(subject)],
        ))
        for key in collected:
            collected[key].append(result[key])
        if SAVE_EVOKEDS:
            for method, responses in result["evokeds"].items():
                for condition, response in zip(CONDITION_ORDER, responses):
                    response.comment = condition
                evoked_outputs[f"{subject}_{method}-ave.fif"] = responses
        recording_files.append(result["reviewed_file"])
        recording_files.append({"role": "P3a continuous EEG", "subject": subject,
                                "file": path.name, "sha256": digest})
        print(f"Completed {position}/{len(participant)}: {subject}")

    combined = {key: pd.concat(parts, ignore_index=True) for key, parts in collected.items()}
    table_s1, agreement, group_tests, contrasts = overlap_statistics(combined["participant_contrasts"], participant)
    settings = {
        "expected_n_participants": EXPECTED_N_PARTICIPANTS,
        "condition_order": CONDITION_ORDER, "event_to_condition": EVENT_TO_CONDITION,
        "target_event_ids": TARGET_EVENT_IDS, "saccade_event_ids": SACCADE_EVENT_IDS,
        "eog_anode": EOG_ANODE, "eog_cathode": EOG_CATHODE, "sign_right": SIGN_RIGHT,
        "heog_filter_hz": [FILTER_LOW_HZ, FILTER_HIGH_HZ], "detector": DETECTOR_PARAMETERS,
        "automatic_response_window_s": RESPONSE_WINDOW, "search_lower_bound_inclusive": True,
        "reviewed_response_latency_window_ms": CLEAN_LATENCY_WINDOW_MS,
        "response_selection_source": "Final manually reviewed latency and direction",
        "match_tolerance_ms": MATCH_TOLERANCE_MS,
        "epoch_interval_s": [EPOCH_TMIN, EPOCH_TMAX], "baseline_s": BASELINE,
        "eeg_reject_peak_to_peak_v": EEG_REJECT["eeg"], "roi": ERP_ROI, "roi_window_s": ERP_WINDOW,
        "regression_tstep_s": REGRESSION_REJECT_TSTEP_S, "regression_decim": 1, "regression_solver": "masked_cholesky",
        "epoch_reject_by_annotation": True, "regression_reject_by_annotation": True,
        "regression_mask_shared": True,
        "stored_channel_types_preserved": True, "self_score": "self_comp from notebook 03",
        "wilcoxon": {"alternative": "two-sided", "zero_method": "wilcox", "correction": False, "method": "auto"},
        "p_values": "Unadjusted", "save_evokeds": SAVE_EVOKEDS,
    }
    tables = {
        "Table_S1": table_s1, "Method_agreement": agreement,
        "Group_probability_tests": group_tests, "Participant_contrasts": contrasts,
        "Condition_amplitudes": combined["condition_amplitudes"],
        "Main_ERP_comparison": pd.concat(comparisons, ignore_index=True),
        "Trial_QC": combined["trial_qc"], "Continuous_saccades": combined["saccade_events"],
        "Processing_QC": combined["processing_qc"], "Warnings": combined["warnings"],
        "Software_versions": SOFTWARE_VERSIONS.copy(),
        "Input_files": pd.concat([input_files, pd.DataFrame(recording_files)], ignore_index=True),
        "Analysis_settings": pd.DataFrame([{"setting": key, "value": json.dumps(value, ensure_ascii=False)}
                                            for key, value in settings.items()]),
    }
    return tables, evoked_outputs


if RUN_ANALYSES:
    tables, evoked_outputs = run_overlap_analysis()
else:
    tables, evoked_outputs = None, {}
    print("Configure the input paths and enable RUN_ANALYSES to analyze study recordings.")

## 7. Inspect results

Review correlations, group tests, and the agreement check before export. `Main_ERP_comparison` reports overlap-control minus primary-analysis means and counts, as well as exact equality of retained target sample sets. A nonzero discrepancy needs investigation even if the participant-level correlation is high. Processing warnings remain available in full in the workbook.

Trial_QC reports reviewed-response exclusions and automatic/manual agreement. Processing_QC reports the shared regression mask and input filter metadata. Metadata are recording-wide summaries and do not establish the filter history of individual EOG channels.

In [ ]:
if RUN_ANALYSES:
    for name in ["Table_S1", "Method_agreement", "Group_probability_tests"]:
        print(name)
        display(tables[name])
    comparison = tables["Main_ERP_comparison"]
    disagreement = (~comparison["amplitude_agrees"].astype(bool)
                    | comparison["count_difference"].ne(0)
                    | ~comparison["retained_samples_agree"].astype(bool))
    print("Participant-condition comparisons requiring review:", int(disagreement.sum()))
    if disagreement.any():
        display(comparison.loc[disagreement])
    print("Captured processing warnings:", len(tables["Warnings"]))
    if not tables["Warnings"].empty:
        display(tables["Warnings"].head(20))
    display(tables["Processing_QC"][["subject", "n_targets", "n_continuous_saccades",
        "n_epochs_all", "n_epochs_selected", "n_shifted_saccade_predictors", "n_bad_annotations"]])
    qc = tables["Trial_QC"]
    matched = qc["matched_within_tolerance"].astype(bool)
    print("Reviewed detections:", int(qc["reviewed_detected"].sum()))
    print("Matched within tolerance:", int(matched.sum()))
    print("Median onset error among matches (ms):", qc.loc[matched, "onset_error_ms"].median())
    print("Direction disagreements among matches:", int(qc["matched_direction_disagrees"].sum()))
    print("Rows sharing a matched automatic event:", int(qc["shared_matched_auto_event"].sum()))
    print("Incomplete reviewed annotations:", int(qc["incomplete_annotation"].sum()))
    print("Reviewed response exclusions before EEG artifact rejection:")
    display(qc.loc[~qc["response_selected"], "response_exclusion_reason"].value_counts())
    print("Regression artifact mask:")
    display(tables["Processing_QC"][[
        "subject", "n_regression_bad_samples", "n_regression_amplitude_samples",
        "n_regression_excluded_samples", "n_regression_eligible_samples",
    ]])
    print("Input filter metadata:")
    display(tables["Processing_QC"].groupby(["sfreq_hz", "input_highpass_hz", "input_lowpass_hz"]).size())
    unusual_filter = tables["Processing_QC"].loc[
        ~np.isclose(tables["Processing_QC"]["input_lowpass_hz"], 30.0)
    ]
    if not unusual_filter.empty:
        display(unusual_filter[["subject", "input_highpass_hz", "input_lowpass_hz"]])

## 8. Export

Enable `EXPORT_OUTPUTS` to write `saccade_overlap_control.xlsx`. Optionally save all four sets of participant target waveforms to `evoked/`. Export checks all destination files before writing and stops if any already exist unless `OVERWRITE_OUTPUTS=True`.

The workbook contains participant-level data and detected eye-movement timings. Keep these outputs in private storage; the public repository contains the analysis code and documentation.

In [ ]:
def export_overlap_results(tables, evoked_outputs):
    if tables is None:
        raise ValueError("Run the cohort analysis before exporting.")
    workbook = OUTPUT_DIR / "saccade_overlap_control.xlsx"
    destinations = [workbook] + [OUTPUT_DIR / "evoked" / name for name in evoked_outputs]
    existing = [str(path) for path in destinations if path.exists()]
    if existing and not OVERWRITE_OUTPUTS:
        raise FileExistsError("Output files already exist; choose another output directory or explicitly enable overwrite: "
                              + ", ".join(existing))
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(workbook, engine="openpyxl") as writer:
        for name, table in tables.items():
            table.to_excel(writer, sheet_name=name, index=False)
            worksheet = writer.sheets[name]
            worksheet.freeze_panes = "A2"
            worksheet.auto_filter.ref = worksheet.dimensions
    if evoked_outputs:
        (OUTPUT_DIR / "evoked").mkdir(exist_ok=True)
        for name, responses in evoked_outputs.items():
            mne.write_evokeds(OUTPUT_DIR / "evoked" / name, responses,
                              overwrite=OVERWRITE_OUTPUTS, verbose="error")
    print("Saved:", workbook)
    if evoked_outputs:
        print("Participant waveform files:", len(evoked_outputs))
    return destinations


if RUN_ANALYSES and EXPORT_OUTPUTS:
    exported_files = export_overlap_results(tables, evoked_outputs)
else:
    print("Export disabled.")

## References

Smith, N. J., & Kutas, M. (2015). Regression-based estimation of ERP waveforms: I. The rERP framework. *Psychophysiology, 52*(2), 157–168. https://doi.org/10.1111/psyp.12317

Ehinger, B. V., & Dimigen, O. (2019). Unfold: An integrated toolbox for overlap correction, non-linear modeling, and regression-based EEG analysis. *PeerJ, 7*, e7838. https://doi.org/10.7717/peerj.7838